# Scientific Programming with Python 
## (Winter 2025/26)

# Session 03: Numpy & MatPlotLib

* Organization
* Some more python
* Finishing Numpy
* MatPlotLib

# Organization


## Exercise groups

* group sizes will be incread to 6 persons
  - split your group into two subgroups
  - each subgroup meets the tutor every 2nd week
* work in groups, combine your solutions
  - upload one solution per subgroup
  - into the designated group folde 
* from now on, sheets are handed out every second week
  - do at least n-1 of the sheets for qualifying to the final project 

## Tutor meetings

There will be initial tutor meetings this week. Meetings can be online or onsite - discuss with your tutor.  For in-person meetings, we have the following rooms:
* Wed., 12:00 - 14:00  Room: 93/E01
* Wed., 14:00 - 16:00  Room: 93/E12
* Thu., 12:00 - 14:00  Room: 35/E22
online meetings on Friday in BBB

At the meeting:
* present your solutions to the exercises
* questions & answers
* feedback

# New exercise sheet

* next week

# The Python Data Model

We have already seen math-operators on numbers:

In [6]:
# Standard math operators work as expected on numbers
a = 2
b = 3

print('a + b = ', a + b)
print('a - b = ', a - b)
print('a * b = ', a * b)
print('a ** b = ', a ** b)  # a to the power of b (a^b is a bit-wise XOR!)
print('a / b = ', a / b)
print('a // b = ', a // b)  # Floor division 
print('b % a = ', b % a)    # Modulo operator (divide, return remainder)

a + b =  5
a - b =  -1
a * b =  6
a ** b =  8
a / b =  0.6666666666666666
a // b =  0
b % a =  1


And we've also seen these operators on strings!

In [7]:
print('hello' + 'world')
print('hello' * 3)

helloworld
hellohellohello


So how does Python know *which one of these* it is supposed to use?  
  
  
Under the hood, most Python syntax is just *syntatic sugar* for method calls. 

So if you call...

In [8]:
mylist = [1, 2, 3]
len(mylist)

3

What Python makes of this is actually:

In [9]:
mylist.__len__()

3

So, under the hood of python there's basically this:

In [10]:
def len(obj):
    return obj.__len__()

If you know the methods that are implicitly called by the common syntax you can design objects that beautifully integrate with the language.

Another example:

In [11]:
3 + 3 

6

is in fact doing 

In [1]:
(3).__add__(3)

6

Let's find out how we can make objects that behave nicely with the rest of the language. As an example we look at a class for representing triples of numeric values.

In [13]:
class Triple:
    def __init__(self, num1, num2, num3):
        self.nums = num1, num2, num3

In [14]:
Triple(1, 2, 3)

The string representation of our class is not really informative. We can fix this by implementing `__repr__`.

In [15]:
class Triple:
    def __init__(self, num1, num2, num3):
        self.nums = num1, num2, num3
        
    def __repr__(self):
        """A string representation for inspecting objects at runtime."""
        return "Triple(" + str(self.nums[0]) + ", " + str(self.nums[1]) + ", " + str(self.nums[2]) + ")"
    
Triple(1, 2, 3)

Triple(1, 2, 3)

To make addition between `Triple`s possible, we have to implement `__add__`. We define the addition `Triple`s just as the elementwise addition of the three numbers.  

In [16]:
class Triple:
    def __init__(self, num1, num2, num3):
        self.nums = num1, num2, num3
        
    def __repr__(self):
        """A string representation for inspecting objects at runtime."""
        return "Triple(" + str(self.nums[0]) + ", " + str(self.nums[1]) + ", " + str(self.nums[2]) + ")"
    
    def __add__(self, other):
        num1 = self.nums[0] + other.nums[0]
        num2 = self.nums[1] + other.nums[1]
        num3 = self.nums[2] + other.nums[2]
        return Triple(num1, num2, num3)
        
    
a = Triple(1, 2, 3)
b = Triple(2, 3, 4)

Because we implemented `__add__` we can add triples with the plus operator.
The following three expressions are all the same! The first one is the fast way to write it, which 
internally maps to the second, which internally maps to the third!


In [17]:
print(a + b)
print(a.__add__(b))
print(Triple.__add__(a, b))

Triple(3, 5, 7)
Triple(3, 5, 7)
Triple(3, 5, 7)


In [ ]:
class Triple:
    def __init__(self, num1, num2, num3):
        self.nums = num1, num2, num3
        
    def __repr__(self):
        """A string representation for inspecting objects at runtime."""
        return "Triple(" + str(self.nums[0]) + ", " + str(self.nums[1]) + ", " + str(self.nums[2]) + ")"
    
    def __add__(self, other):
        if isinstance(other, Triple):
            num1 = self.nums[0] + other.nums[0]
            num2 = self.nums[1] + other.nums[1]
            num3 = self.nums[2] + other.nums[2]
            return Triple(num1, num2, num3)
        elif isinstance(other, int):
            return Triple(self.nums[0]+other, self.nums[1]+other, self.nums[2]+other)
        else:
            return NotImplemted

In [ ]:
a = Triple(1, 2, 3)
b = Triple(2, 3, 4)

In [19]:
a + 1

Triple(2, 3, 4)

Now it would be nice to enable addition between triples and scalar numbers by just elementwise adding the scalar to all three triple values. But the expression

In [20]:
1 + Triple(1, 2, 3)

TypeError: unsupported operand type(s) for +: 'int' and 'Triple'

is interpreted as 

In [21]:
int.__add__(1, Triple(1, 2, 3))

NotImplemented

which returns the special value `NotImplemented`. If a binary operaton does not work when called on the first operand does not work, Python tries to invert the order of operands, calling `__radd__` on the other. If this does not work either a `TypeError` is raised.

By implementing `__radd__` we can make scalar addition work without changing the behaivor of the `int`s.

In [22]:
class Triple:
    def __init__(self, num1, num2, num3):
        self.nums = num1, num2, num3
        
    def __repr__(self):
        """A string representation for inspecting objects at runtime."""
        return "Triple(" + str(self.nums[0]) + ", " + str(self.nums[1]) + ", " + str(self.nums[2]) + ")"
    
    def __add__(self, other):
        if isinstance(other, Triple):
            num1 = self.nums[0] + other.nums[0]
            num2 = self.nums[1] + other.nums[1]
            num3 = self.nums[2] + other.nums[2]
            return Triple(num1, num2, num3)
        elif isinstance(other, int):
            return Triple(self.nums[0]+other, self.nums[1]+other, self.nums[2]+other)
        else:
            return NotImplemted
    
    def __radd__(self, other):
        num1 = self.nums[0] + other
        num2 = self.nums[1] + other
        num3 = self.nums[2] + other
        return Triple(num1, num2, num3)

In [23]:
1 + Triple(1, 2, 3)

Triple(2, 3, 4)

In [24]:
Triple(1, 5, 7) + 2

Triple(3, 7, 9)

## Extending arrays

In [ ]:
import numpy as np

### Adding new dimensions with `np.newaxis`

Instead of `np.newaxis`, `None` can be used.

In [ ]:
one_dim_arr = np.arange(5)
one_dim_arr, one_dim_arr.shape

In [ ]:
two_dim_arr = one_dim_arr[np.newaxis, :]
two_dim_arr, two_dim_arr.shape

In [ ]:
two_dim_arr = one_dim_arr[:, np.newaxis, None]
two_dim_arr, two_dim_arr.shape

Adding new dimensions is useful for example when Tensorflow is used to batch-inputs, but you want to provide a single datapoint for prediction:

In [ ]:
one_dim_arr[:, None]

### Removing dimensions

``arr.squeeze()`` removes dimensions of size 1:

In [ ]:
one_dim_arr = np.arange(5)
two_dim_arr = one_dim_arr[np.newaxis, :]
two_dim_arr, two_dim_arr.shape

In [ ]:
two_dim_arr.squeeze(), two_dim_arr.squeeze().shape

In [ ]:
a = np.arange(5).reshape(1, -1, 1, 1)
a

In [ ]:
a.squeeze()

### Combining arrays
There are many ways to combine existing arrays, like `np.append`, `np.concatenate` and `np.stack`. However, these operations always require the whole array to be copied. Therefore, it often makes more sense to allocate an array of the size you need later upfront and then just fill the respective parts.

In [ ]:
np.concatenate((np.arange(10), np.arange(10)[::-1]))

A quick and easy way to combine scalars and arrays is using ``np.r_``, with the desired arrays, lists, or numbers in square brackets:

In [ ]:
np.r_[2, 2, 2, np.arange(10), np.arange(10)[::-1], [0, 1, 2]]

``np.append`` uses concatenation internally:

In [ ]:
np.append(np.arange(10), np.arange(10))

For higher-dimensional arrays, other functions are useful:

In [ ]:
np.stack((np.arange(10), np.arange(10)))

There are also the functions ``np.vstack`` (row-wise-stacking) and ``np.hstack`` (column-wise-stacking):
* hstack is equivalent to concatenation along the second axis, except for 1-D arrays where it concatenates along the first axis
* vstack is equivalent to concatenation along the first axis after 1-D arrays of shape (N,) have been reshaped to (1,N).

In [ ]:
two_dim_arr = np.arange(16).reshape(4, -1)
two_dim_arr_2 = np.arange(16).reshape(4, -1) + 16
two_dim_arr, two_dim_arr_2

In [ ]:
np.hstack((two_dim_arr, two_dim_arr_2))

In [ ]:
np.vstack((two_dim_arr, two_dim_arr_2))

## Random values and random operations

In [ ]:
import numpy as np

### random.seed

If a random seed is set, the random-number-generator re-uses the same numbers over and over again. This is very useful for testing, but of course this takes any randomness out of anything, so it should not be used in final code:

In [ ]:
for _ in range(5):
    rng = np.random.default_rng(seed=0)
    print(rng.random(5))

To disable it, set a random seed of ``None``, in which case numpy generates random numbers using your system-randomness (or the system-time, ...)

In [ ]:
for _ in range(5):
    rng = np.random.default_rng(seed=None)
    print(rng.random(5))

### Shuffling arrays

``rng.shuffle`` shuffles an array among the first dimension. That means, a one-dimensional is completely shuffled, whereas for multidimensional arrays, the arrays from the second dimension on remain intact.

In [ ]:
a = np.arange(10)
rng.shuffle(a)
a

Shuffling a multidimensional array just shuffles the first axis:

In [ ]:
a = np.arange(9).reshape(3, 3)
rng.shuffle(a)

To shuffle the array completely, you can flatten it and afterwards reshape it to its original shape:

In [ ]:
b = a.flatten()
rng.shuffle(b)
a = b.reshape(a.shape)
a

Note that ``np.shuffle`` shuffles the array in-place. To return a permutation, you'd use ``np.permutation``:

In [ ]:
a = np.arange(9).reshape(3, 3)
a

In [ ]:
rng.permutation(a)

In [ ]:
a

If you are dealing with different arrays you want to shuffle while keeping them matched to each other, it is more useful to shuffle the indices instead:

In [ ]:
a = np.arange(9)+1
b = a ** 2
np.vstack((a, b))

In [ ]:
order = rng.permutation(a.shape[0])
np.vstack((a[order], b[order]))

### random.choice

`rng.choice` generates a sub-array from a given 1D-array:

In [ ]:
a = np.arange(10)
rng.choice(a, size=5)

In [ ]:
rng.choice(a, size=5, replace=False)

In [ ]:
rng.choice(a, size=11, replace=False)  # this will fail

You can also specify probabilities which which to take certain elements. To generate another array where roughly a quarter of elements are True, you can use:

In [ ]:
rng.choice(np.array([True, False]), size=(5,5), p=[0.25, 0.75])

## The linear algebra module

Numpy provides a module for linear algebra: `numpy.linalg`
- Linear algebra routines for NumPy arrays (ndarray), backed by BLAS/LAPACK.
- Works with floats/complex numbers; integer inputs are cast to float.
- Operates on 2D arrays as matrices; vectors are 1D.
- Always check shapes and dtypes before calling linalg routines.

In [ ]:
import numpy as np

In [ ]:
A = np.array([[3.0, 2.0],
              [1.0, 2.0]])
b = np.array([2.0, 0.0])

print("A shape/dtype:", A.shape, A.dtype)
print("b shape/dtype:", b.shape, b.dtype)

Solving linear systems
- Solve $Ax = b$ with `np.linalg.solve` for square, nonsingular $A$.

In [ ]:
x = np.linalg.solve(A, b)
res = A @ x - b
print("x:", x)
print("residual norm:", np.linalg.norm(res))

Least squares for linear systems for over/underdetermined problems
- Least squares fit y ≈ a + b*x: `np.linalg.lstsq` minimizes $||Ax − b||^2$.
- Check residuals to assess fit quality.

In [ ]:
xdata = np.array([1.0, 2.0, 3.0])
ydata = np.array([1.0, 2.0, 2.7])
X = np.column_stack([np.ones_like(xdata), xdata])  # columns: [1, x]
coef, residuals, rank, svals = np.linalg.lstsq(X, ydata, rcond=None)
a_hat, b_hat = coef
print("least-squares parameters a, b:", a_hat, b_hat)
print("residual sum of squares:", residuals if residuals.size else 0.0)

Key decompositions (eig/eigh, SVD, QR)
- Eigen-decomposition: `np.linalg.eig` (general), `np.linalg.eigh` (symmetric/Hermitian).
- SVD: `np.linalg.svd` gives U, s, VT; useful for rank, pseudoinverse, PCA.
- QR: `np.linalg.qr` for orthogonal-triangular factorization (useful in solves/LS).

Symmetric eigen-decomposition

In [ ]:
S = np.array([[2.0, 1.0],
              [1.0, 2.0]])
w, V = np.linalg.eigh(S)  # w sorted ascending
print("eigenvalues:", w)
print("check eigenpair 0:", np.allclose(S @ V[:, 0], w[0] * V[:, 0]))

Singular value decomposition (SVD):

In [ ]:
M = np.array([[3.0, 1.0, 1.0],
              [-1.0, 3.0, 1.0]])
U, s, VT = np.linalg.svd(M, full_matrices=False)

check the reconstruction:

In [ ]:
M_rec = U @ np.diag(s) @ VT
print("SVD reconstruction close:", np.allclose(M, M_rec))

(Reduced) QR decomposition:

In [ ]:
Q, R = np.linalg.qr(M, mode="reduced")
print("QR check:", np.allclose(Q @ R, M), "Q orthogonal:", np.allclose(Q.T @ Q, np.eye(Q.shape[0])))

## Numpy print-options

In its standard options, numpy prints numbers only up to a certain accuracy, or consolidates the printing of arrays. ``np.set_printoptions`` can change that:

In [ ]:
rand_arr = rng.random((5,3))
rand_arr

In [ ]:
np.set_printoptions(precision=3)
rand_arr

In [ ]:
rand_arr = rand_arr/1e3
rand_arr

In [ ]:
np.set_printoptions(suppress=True, precision=6)
rand_arr

In [ ]:
arr = np.arange(90)
arr

In [ ]:
np.set_printoptions(threshold=10)
arr

## Further Readings

Official Numpy Page: [https://numpy.org/](https://numpy.org/)

NumPy chapter from Jake VanderPlas's [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/02.00-introduction-to-numpy.html)

[Video tutorial from Scipy 2017](https://youtu.be/lKcwuPnSHIQ)

In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo('lKcwuPnSHIQ')

# Plotting with Matplotlib

Credits to Ben Root and the [Anatomy of Matplotlib](https://github.com/matplotlib/AnatomyOfMatplotlib) tutorial as well as to  Seyed Arman Taghizadeh Motlagh for the providing code on which this document is based.

## Introduction

Matplotlib is a library for producing publication-quality figures. mpl (for short) was designed from the beginning to serve two purposes. 
* First, allow for interactive, cross-platform control of figures and plots, and
* second, to make it very easy to produce static raster or vector graphics files without the need for any GUIs.

Furthermore, mpl -- much like Python itself -- gives the developer complete control over the appearance of their plots, while still being very usable through a powerful defaults system.

## Online Documentation

The [matplotlib.org](http://matplotlib.org) project website is the primary online resource for the library's documentation. It contains [examples](http://matplotlib.org/examples/index.html), [FAQs](http://matplotlib.org/faq/index.html), [API documentation](http://matplotlib.org/api/index.html), and, most importantly, the [gallery](http://matplotlib.org/gallery.html).


# Plotting Functions

Matplotlib has a number of different plotting functions -- many more than we'll cover here, in fact. There's a more complete list in the pyplot documentation, and Matplotlib gallery is a great place to get examples of all of them.  

However, a full list and/or the gallery can be a bit overwhelming at first. Instead we'll condense it down and give you a look at some of the ones you're most likely to use, and then go over a subset of those in more detail.

Here's a simplified visual overview of matplotlib's most commonly used plot types.  Let's browse through these, and then we'll go over a few in more detail. Clicking on any of these images will take you to the code that generated them. We'll skip that for now, but feel browse through it later.

## The Basics: 1D series/points

<a  href="examples/plot_example.py"><img src="images/plot_example.png"></a>

<a href="examples/scatter_example.py"><img src="images/scatter_example.png"></a>

<a href="examples/bar_example.py"><img src="images/bar_example.png"></a>

<a href="examples/fill_example.py"><img src="images/fill_example.png"></a>

## 2D Arrays and Images
<a href="examples/imshow_example.py"><img src="images/imshow_example.png"></a>

<a href="examples/pcolor_example.py"><img src="images/pcolor_example.png"></a>

<a href="examples/contour_example.py"><img src="images/contour_example.png"></a>

## Vector Fields
<a href="examples/vector_example.py"><img src="images/vector_example.png"></a>

## Data Distributions
<a href="examples/statistical_example.py"><img src="images/statistical_example.png"></a>

## Input Data: 1D Series

Let's move on to a couple of common plot types.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## Basic Plotting

`plot` draws points with lines connecting them.  `scatter` draws unconnected points, optionally scaled or colored by additional variables.

As a basic example:

In [ ]:
fig = plt.figure()
plt.plot([1, 2, 3, 4], [10, 20, 25, 35])
plt.show()

Multiple calls to the plot function will result in multiple plots. This works with many functions in mpl (plot, scatter, bar, etc.) and assigns different colors to the different calls.

In [ ]:
fig = plt.figure()
plt.plot([10, 20, 25, 35])
plt.plot([65, 34, 12, -12])
plt.show()

### Bar Plots: `ax.bar(...)` and `ax.barh(...)`

Bar plots are one of the most common plot types.  Matplotlib's `ax.bar(...)` method can also plot general rectangles, but the default is optimized for a simple sequence of x, y values, where the rectangles have a constant width.  There's also `ax.barh(...)` (for horizontal), which makes a constant-height assumption instead of a constant-width assumption.

### Simple bar plot

In [ ]:
np.random.seed(2)
x = np.arange(5)
y = np.random.random(5) * 2

fig = plt.figure()
plt.bar(x, y)
plt.show()

In [ ]:
x

In [ ]:
y

Adding errorbars...

In [ ]:
np.random.seed(2)
x = np.arange(5)
y = np.random.random(5) * 2
error = y * 0.1

fig = plt.figure()
plt.bar(x, y, yerr=error)
plt.show()

If we have negative values, we can use `axhline` to draw an axis "spine" to mark the zero line.

In [ ]:
np.random.seed(1)
x = np.arange(5)
y = np.random.randn(5)

fig = plt.figure()
plt.bar(x, y)
plt.axhline(y=0, color='black', linewidth=2)
plt.show()

Matplotlib plotting methods return an `Artist` or a sequence of artists.  Anything you can see in a Matplotlib figure/axes/etc is an `Artist` of some sort. Most of the time, you will not need to retain these returned objects. You will want to capture them for special customizing that may not be possible through the normal plotting mechanism.

Let's re-visit that last example and modify what's plotted.  In the case of `bar`, a container artist is returned, so we'll modify its contents instead of the container itself (thus, `for bar in vert_bars`).

In [ ]:
fig = plt.figure()
vert_bars = plt.bar(x, y) # Store the output of the call to .bar

for bar, height in zip(vert_bars, y):
    if height < 0:
        bar.set(color='red')

plt.axhline(y=0, color='black', linewidth=2)
plt.show()

Looking at the artist returned by `ax.bar` we can see that they are all plain rectangles.

In [ ]:
for bar in vert_bars:
    print(bar)

Remark: we could have also done this with two separate calls to `plt.bar` and numpy boolean indexing:

In [ ]:
fig = plt.figure()
plt.bar(x[y>0], y[y>0])
plt.bar(x[y<0], y[y<0], color='red')
plt.axhline(y=0, color='black', linewidth=2)
plt.show()

### Filled Regions: `fill(x, y)`, `fill_between(...)`, etc

Of these functions, `plt.fill_between(...)` is probably the one you'll use the most often.  In its most basic form, it fills between the given y-values and 0:

In [ ]:
np.random.seed(1)
y = np.random.randn(100).cumsum()
x = np.linspace(0, 10, 100)

fig = plt.figure()
plt.fill_between(x, y, color='lightblue')
plt.show()

In [ ]:
x = np.linspace(0, 10, 200)
y1 = 2 * x + 1
y2 = 3 * x + 1.2
y_mean = 0.5 * x * np.cos(2*x) + 2.5 * x + 1.1

fig = plt.figure()

# Plot the envelope with `fill_between`
plt.fill_between(x, y1, y2, color='yellow')

# Plot the "centerline" with `plot`
plt.plot(x, y_mean, color='black')

plt.show()

## Input Data: 2D Arrays or Images

There are several options for plotting 2D datasets.  `imshow`, `pcolor`, and `pcolormesh` have a lot of overlap, at first glance.  The image above is meant to clarify that somewhat.


In short, `imshow` can interpolate and display large arrays very quickly, while `pcolormesh` and `pcolor` are much slower, but can handle flexible (i.e. more than just rectangular) arrangements of cells.

We won't dwell too much on the differences and overlaps here.  They have overlapping capabilities, but different default behavior because their primary use-cases are a bit different (there's also `matshow`, which is `imshow` with different defaults).  

Instead we'll focus on what they have in common.

`imshow`, `pcolor`, `pcolormesh`, `scatter`, and any other Matplotlib plotting methods that map a range of data values onto a colormap will return artists that are instances of `ScalarMappable.`  In practice, what that means is a) you can display a colorbar for them, and b) they share several keyword arguments.

### Displaying 2d data with `imshow`

In [ ]:
arr_2d = np.arange(9).reshape((3, 3))
arr_2d

In [ ]:
fig = plt.figure()
plt.imshow(arr_2d)
plt.show()

`imshow` is used most of the times to display images.

In [ ]:
# Read images into numpy array. Usually imageio would be used here.
# see https://imageio.github.io/
img = plt.imread("images/rocket.png")

fig = plt.figure()
plt.imshow(img)
plt.axis("off")
plt.show()

For visualizing matrices, `matshow` provides better defaults, e.g. axis labelling.

In [ ]:
fig = plt.figure()
plt.matshow(arr_2d)
plt.show()

### Colorbars

Just seeing the colors does not necessarily tell us something about the values beneath. Let's add a colorbar to the figure to display what colors correspond to values of `data` we've plotted. 

In [ ]:
fig = plt.figure()
im = plt.imshow(arr_2d)
fig.colorbar(im)
plt.show()

In [ ]:
arr_2d

The colorbar is usually placed next to the plot (from which it "steels" some space).  It is also possible to create an own `Axes` object to place the colorbar (more on `Axes` next week).

In [ ]:
fig = plt.figure()
im = plt.imshow(arr_2d)
cax = fig.add_axes([0.27, 0.8, 0.5, 0.05])
fig.colorbar(im, cax=cax, orientation='horizontal')
plt.show()

### Shared parameters for `imshow`, `pcolormesh`, `contour`, `scatter`, etc
  
  As we mentioned earlier, any plotting method that creates a `ScalarMappable` will have some common kwargs.  The ones you'll use the most frequently are:
  
  * `cmap` : The colormap (or name of the colormap) used to display the input.  (We'll go over the different colormaps in the next section.)
  * `vmin` : The minimum data value that will correspond to the "bottom" of the colormap (defaults to the minimum of your input data).
  * `vmax` : The maximum data value that will correspond to the "top" of the colormap (defaults to the maximum of your input data).
  * `norm` : A `Normalize` instance to control how the data values are mapped to the colormap. By default, this will be a linear scaling between `vmin` and `vmax`, but other norms are available (e.g. `LogNorm`, `PowerNorm`, etc).
  
`vmin` and `vmax` are particularly useful.  Quite often, you'll want the colors to be mapped to a set range of data values, which aren't the min/max of your input data. For example, you might want a symmetric ranges of values around 0.

As an example of that, let's use a divergent colormap on some example data. Note how the colormap is **not** centered at zero.

In [ ]:
from matplotlib.cbook import get_sample_data
data = get_sample_data('axes_grid/bivariate_normal.npy')

fig, ax = plt.subplots()
im = ax.imshow(data, cmap='seismic')
fig.colorbar(im)
plt.show()

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(data, cmap='seismic', vmin=-2, vmax=2)
fig.colorbar(im)
plt.show()

## Scatter for n-dimensional data
`scatter` allows to map several dimensions to different aesthetics such as x-postion, color, size and shape.

In [ ]:
n = 50
x1 = np.random.random(n)
x2 = np.random.random(n) * 50
x3 = np.random.random(n)
y = x1 + x2 + x3

fig = plt.figure()
sc = plt.scatter(x=x1, y=y, s=x2, c=x3, marker='o')
fig.colorbar(sc)
plt.show()

There are lots of different markers mpl supports which can help to emphasize different distributions. Have a look at https://matplotlib.org/3.2.1/api/markers_api.html#module-matplotlib.markers to get a list of marker options.

In [ ]:
num = 100
x1 = np.random.normal(-1, 2, num)
y1 = np.random.normal(3, 4, num)
x2 = np.random.normal(3, 2, num)
y2 = np.random.normal(1, 1, num)

fig, ax = plt.subplots()
ax.scatter(x1, y1, marker='.')
ax.scatter(x2, y2, marker='x')
plt.show()

## Visualizing statistical distributions

Draw samples from a normal distribution.

In [ ]:
μ = 0
σ = 1
num_samples = 1000
dist = np.random.normal(μ, σ, num_samples)

**Side note on unicode characters as variables.** Since Python 3 we can use any unicode character such as μ and σ as variables. This can make sense in scientific programming if strong naming conventions exist. A good example is mean and standard deviation of a normal distribution. To easily obtain common characters you can use latex style and type e.g. `\mu` followed by <kbd>tab</kbd> to obtain μ. However, do not overuse this. Clearly named characters are often easier to read.

### Histograms
Histograms a great way to visualize univariate distributions. The data is divided into equally sized bins. Then we count how many data points fall into each bin. Finally, we draw a bar with the width of the corresponding bin and the height of the count.

The standard histogram looks like this.

In [ ]:
fig = plt.figure()
plt.hist(dist)
plt.show()

We can control the number of bins.

In [ ]:
fig = plt.figure()
plt.hist(dist, bins=500)
plt.show()

Or let it be automatically determined.

In [ ]:
fig = plt.figure()
plt.hist(dist, bins='auto')
plt.show()

Using `density=True` will create a normalized histogram that can be interpreted as a probability density. 

In [ ]:
fig = plt.figure()
plt.hist(dist, bins='auto', density=True)
plt.show()

### Boxplots
Boxplots are another standard way to summarize univariate distributions. They give a compact visual description of important *summary statistics*. 
A box is drawn at the 25% and 75% quantile, that is where most of the data is. Additionally, the median is marked by a line inside the box.
The *whiskers* extend 1.5 times the *inter quartile range* beyond the quartiles. Every point beyond that is drawn individually as a *flier* or *outlier*. See also https://flowingdata.com/2008/02/15/how-to-read-and-use-a-box-and-whisker-plot/ for a nice illustration.

In [ ]:
fig = plt.figure()
plt.boxplot(dist)
plt.show()

`boxplot` can also be used to display several distributions at once.

In [ ]:
import reprlib # This is for obtaining printable versions of large data sets.
means = [0, -1, 2.5, 4.3, -3.6]
sigmas = [1.2, 5, 3, 1.5, 2]
# Each distribution has a different number of samples.
nums = [150, 1000, 100, 200, 500]

dists = [np.random.normal(*args) for args in zip(means, sigmas, nums)]
reprlib.repr(dists)

In [ ]:
fig = plt.figure()
plt.boxplot(dists)
plt.show()

### Violinplots
Violinplots are a third common way to visualize distributions. For violinplots a *kernel density estimate* is computed for the whole range of data. This gives a smooth estimate of the probabiliy density function underlying the data.
The `violinplot`function behaves similar to `boxplot`. For further information, have a look at https://en.wikipedia.org/wiki/Violin_plot.

In [ ]:
fig = plt.figure()
plt.violinplot(dist)
plt.show()

In [ ]:
fig = plt.figure()
plt.violinplot(dists)
plt.show()

### Pie charts
Pie charts a well known way to visualize categorical distributions.

In [ ]:
# Pie chart, where the slices will be ordered and plotted counter-clockwise:
labels = 'Frogs', 'Hogs', 'Dogs', 'Logs'
sizes = [15, 30, 45, 20]

fig = plt.figure()
plt.pie(sizes, labels=labels, autopct='%1.1f%%')

plt.show()

We will cover more advanced and more convenient methods for statistical visualization in a later lecture.

## Annotating plots
Especially for scientific figures, you often want to hightlight the part of a plot that supports your hypothesis. With maplotlib you have the full flexibility to do this using `annotate`. By default it will just add text at certain x, y coordinate.  

In [ ]:
fig = plt.figure()

t = np.arange(0.0, 5.0, 0.01)
s = np.cos(2*np.pi*t)

# Plot a line and add some simple annotations
line, = plt.plot(t, s)
plt.annotate('local maximum', xy=(3, 1))
plt.ylim(-2, 2)
plt.show()

We can also put the text at a different x, y-coordinat using the `xytext` argument.

In [ ]:
fig = plt.figure()

t = np.arange(0.0, 5.0, 0.01)
s = np.cos(2*np.pi*t)

line, = plt.plot(t, s)
plt.annotate('local maximum', xy=(3, 1), xytext=(3.5, 1.5))

plt.ylim(-2, 2)
plt.show()

Now that we have two x, y - coordinates we can connect them using arrows.

In [ ]:
fig = plt.figure()

t = np.arange(0.0, 5.0, 0.01)
s = np.cos(2*np.pi*t)

# Plot a line and add some simple annotations
line, = plt.plot(t, s)
plt.annotate('local maximum', xy=(3, 1), xytext=(3.5, 1.5), arrowprops=dict())

plt.ylim(-2, 2)
plt.show()

If we have to, we can go very fancy on the arrow styles. See https://matplotlib.org/tutorials/text/annotations.html for a detailed overview.

In [ ]:
fig = plt.figure()

t = np.arange(0.0, 5.0, 0.01)
s = np.cos(2*np.pi*t)

# Plot a line and add some simple annotations
line, = plt.plot(t, s)
plt.annotate('local maximum', xy=(3, 1), xytext=(3.5, 1.5),
             arrowprops=dict(arrowstyle='wedge', connectionstyle="angle3", facecolor="black"))

plt.ylim(-2, 2)
plt.show()